In [7]:
import pandas as pd
import sqlite3

cnx = sqlite3.connect("temp.db")

In [17]:
trades_df = pd.read_csv(r"C:\Users\mdtan\Downloads\tradebook-ZJ6671-FO.csv")
trades_df.head()

,symbol,isin,trade_date,exchange,segment,series,trade_type,auction,quantity,price,trade_id,order_id,order_execution_time,expiry_date
0,TITAN24NOV3600CE,NaN,2024-11-04,NSE,FO,NaN,buy,False,175.0,16.95,980325193,2700000049182895,2024-11-04T10:43:42,2024-11-28
1,TITAN24NOV3350CE,NaN,2024-11-04,NSE,FO,NaN,sell,False,175.0,64.80,980326674,2700000049462489,2024-11-04T10:44:18,2024-11-28
2,BANKNIFTY24NOVFUT,NaN,2024-11-05,NSE,FO,NaN,buy,False,15.0,52120.00,836954452,2100000089748298,2024-11-05T13:54:03,2024-11-27
3,BANKNIFTY24NOVFUT,NaN,2024-11-05,NSE,FO,NaN,buy,False,60.0,52120.00,836954432,2100000089748298,2024-11-05T13:54:03,2024-11-27
4,BANKNIFTY24NOVFUT,NaN,2024-11-05,NSE,FO,NaN,buy,False,300.0,52118.00,836954431,2100000089748298,2024-11-05T13:54:03,2024-11-27


In [37]:
pnl_df = pd.read_excel(r"C:\Users\mdtan\Downloads\pnl-ZJ6671.xlsx", skiprows=37)
cols=list(pnl_df.columns)
renamed_cols = list(map(lambda x: x.replace('.', '').replace('&','n').lower().replace(' ', '_'), cols))
pnl_df = pnl_df.drop(columns=['Unnamed: 0']).rename(columns=dict(zip(cols, renamed_cols)))
pnl_df

,symbol,isin,quantity,buy_value,sell_value,realized_pnl,realized_pnl_pct,previous_closing_price,open_quantity,open_quantity_type,open_value,unrealized_pnl,unrealized_pnl_pct
0,ASIANPAINT24DEC2320PE,NaN,8000,106400.000,363200.000,256800.00,241.3534,0.0,0,NaN,0,0,0.0
1,ASIANPAINT24DEC2380CE,NaN,12000,112160.004,327960.000,215800.00,192.4037,0.0,0,NaN,0,0,0.0
2,ASIANPAINT24DEC2380PE,NaN,8000,668690.000,171400.000,-497290.00,-74.3678,0.0,0,NaN,0,0,0.0
3,ASIANPAINT24DEC2420CE,NaN,12000,174969.996,41769.996,-133200.00,-76.1273,0.0,0,NaN,0,0,0.0
4,ASIANPAINT24DEC2440CE,NaN,8000,85640.000,99600.000,13960.00,16.3008,0.0,0,NaN,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,TITAN24NOV3350CE,NaN,175,2108.750,11340.000,9231.25,437.7593,0.0,0,NaN,0,0,0.0
174,TITAN24NOV3600CE,NaN,175,2966.250,13737.500,10771.25,363.1268,0.0,0,NaN,0,0,0.0
175,TITAN25JAN3350CE,NaN,175,16432.500,8050.000,-8382.50,-51.0117,0.0,0,NaN,0,0,0.0
176,TITAN25JAN3500CE,NaN,350,18340.000,15933.750,-2406.25,-13.1202,0.0,0,NaN,0,0,0.0


In [38]:
pnl_df.to_sql(name="PNL", con=cnx)

178

In [94]:
journalentry_df[['symbol']].to_csv("strategies.csv", index=False)

In [98]:
strategy_df = pd.read_csv("strategies.csv")
strategy_df.to_sql(name="STRATEGY", con=cnx)

ValueError: Table 'STRATEGY' already exists.

In [110]:
journalentry_df = pd.read_sql('''select PNL.symbol,
                trade_type,
                PNL.quantity,
                buy_value,
                sell_value,
                trade_id, entry_time, exit_time, realized_pnl, strategy
            from PNL inner join
                (select symbol, order_id, min(order_execution_time) as entry_time,
                        max(order_execution_time) as exit_time, min(order_id) as trade_id,
                        first_value([trade_type]) over (partition by symbol order by order_execution_time asc) as trade_type
                        from TRADES group by symbol) as T
            on PNL.symbol = T.symbol
            inner join STRATEGY on PNL.symbol = STRATEGY.symbol
            where PNL.quantity <> 0
            order by exit_time desc
            ''', cnx)

,symbol,trade_type,quantity,buy_value,sell_value,trade_id,entry_time,exit_time,realized_pnl,strategy,exit_month
0,NIFTY25FEB22500PE,buy,600,45300.00,60.0,1200000027319166,2025-02-24T10:05:51,2025-02-27T15:25:14,-45240.00,ice-cream bucket,2025-02
1,BANKNIFTY25FEB51200CE,buy,1200,103044.00,3360.0,1800000009373502,2025-02-03T09:39:31,2025-02-25T15:22:16,-99684.00,blacksmith,2025-02
2,BANKNIFTY25FEB49400PE,sell,900,589203.00,527004.0,1800000008854395,2025-02-03T09:38:03,2025-02-25T15:20:01,-62199.00,blacksmith,2025-02
3,BANKNIFTY25FEB47600PE,buy,900,160551.00,48945.0,1800000009472804,2025-02-03T09:40:13,2025-02-25T15:20:00,-111606.00,blacksmith,2025-02
4,BANKNIFTY25FEB48500CE,sell,600,184530.00,222021.0,1700000010783865,2025-02-24T09:27:02,2025-02-25T15:19:50,37491.00,blacksmith,2025-02
...,...,...,...,...,...,...,...,...,...,...,...
171,SBIN24NOV760PE,sell,30000,207000.00,91650.0,2400000148315271,2024-11-05T14:19:10,2024-11-05T14:19:31,-115350.00,None,2024-11
172,SBIN24NOV850CE,sell,30000,232950.00,699000.0,2400000148511402,2024-11-05T14:17:01,2024-11-05T14:17:01,466050.00,None,2024-11
173,SBIN24NOV780PE,buy,30000,150000.00,342000.0,2400000147585202,2024-11-05T14:15:45,2024-11-05T14:15:45,192000.00,None,2024-11
174,SBIN24NOV830CE,buy,30000,1080000.00,363637.5,2400000147586260,2024-11-05T14:15:03,2024-11-05T14:15:07,-716362.50,None,2024-11


In [125]:
journalentry_df['exit_month'] = journalentry_df['exit_time'].apply(lambda x: x[:7])
grp = journalentry_df.groupby(['strategy', 'exit_month'])
grp.groups.keys()

dict_keys([('blacksmith', '2025-01'), ('blacksmith', '2025-02'), ('covered call', '2025-01'), ('ice-cream bucket', '2025-01'), ('ice-cream bucket', '2025-02'), ('iron condor', '2025-01'), ('iron condor', '2025-02'), ('kumo breakout', '2025-01'), ('manual', '2025-02'), ('money in the bank', '2025-01'), ('money in the bank', '2025-02'), (nan, '2024-11'), (nan, '2024-12'), (nan, '2025-01'), (nan, '2025-02')])

In [131]:
grp.get_group(('blacksmith', '2025-02'))

,symbol,trade_type,quantity,buy_value,sell_value,trade_id,entry_time,exit_time,realized_pnl,strategy,exit_month
1,BANKNIFTY25FEB51200CE,buy,1200,103044.0,3360.0,1800000009373502,2025-02-03T09:39:31,2025-02-25T15:22:16,-99684.0,blacksmith,2025-02
2,BANKNIFTY25FEB49400PE,sell,900,589203.0,527004.0,1800000008854395,2025-02-03T09:38:03,2025-02-25T15:20:01,-62199.0,blacksmith,2025-02
3,BANKNIFTY25FEB47600PE,buy,900,160551.0,48945.0,1800000009472804,2025-02-03T09:40:13,2025-02-25T15:20:00,-111606.0,blacksmith,2025-02
4,BANKNIFTY25FEB48500CE,sell,600,184530.0,222021.0,1700000010783865,2025-02-24T09:27:02,2025-02-25T15:19:50,37491.0,blacksmith,2025-02
5,BANKNIFTY25FEB49400CE,sell,600,10200.0,426480.0,1800000008854228,2025-02-03T09:38:05,2025-02-25T15:19:29,416280.0,blacksmith,2025-02


In [70]:
pd.read_sql('''select PNL.symbol, 
                (case when open_quantity_type = 'buy' then open_quantity else -open_quantity end) as quantity, 
            open_value as buy_value, null as sell_value, trade_id, entry_time,
            null as exit_time, null as realized_pnl
            from PNL inner join 
                (select symbol, order_id, min(order_execution_time) as entry_time,
                        max(order_execution_time) as exit_time, min(order_id) as trade_id 
                        from TRADES group by symbol) as T
            on PNL.symbol = T.symbol where PNL.quantity = 0''', cnx)

,symbol,quantity,buy_value,sell_value,trade_id,entry_time,exit_time,realized_pnl
0,BANKNIFTY25MAR48200PE,210,105462,None,1900000097932800,2025-02-27T15:18:39,None,None
1,BANKNIFTY25MARFUT,210,10306191,None,1700000163265578,2025-02-27T15:13:01,None,None
